# 01 Recommendation System
Nearest-neighbor + cluster-backed recommendation experiments.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI Dataset

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


## Similarity Experiments

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans

feature_sets = build_post_feature_sets(df)
metrics = ["cosine", "euclidean", "manhattan"]
k_values = [3, 5, 10, 15]
rows = []
neighbors_cache = {}

for fset, cols in feature_sets.items():
    cat_cols = [c for c in cols if str(df[c].dtype) in ["object", "bool"]]
    num_cols = [c for c in cols if c not in cat_cols]
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    X = pre.fit_transform(df[cols])
    for metric in metrics:
        for k in k_values:
            nn = NearestNeighbors(n_neighbors=min(k + 1, len(df)), metric=metric)
            nn.fit(X)
            dists, idxs = nn.kneighbors(X)
            sims, uplifts = [], []
            for i in range(len(df)):
                neigh = [j for j in idxs[i] if j != i][:k]
                if not neigh:
                    continue
                ds = dists[i][1:1 + len(neigh)] if len(dists[i]) > 1 else dists[i][:len(neigh)]
                sim = np.clip(1 - ds, 0, 1) if metric == "cosine" else 1 / (1 + ds)
                sims.extend(sim.tolist())
                uplifts.append(float(df.iloc[neigh]["engagement_rate"].mean() - df.iloc[i]["engagement_rate"]))
            rows.append({
                "feature_set": fset,
                "metric": metric,
                "k": k,
                "avg_similarity_quality": float(np.mean(sims)) if sims else 0.0,
                "engagement_uplift": float(np.mean(uplifts)) if uplifts else 0.0,
                "interpretability": {"performance-only":0.85,"content-only":0.95,"combined":0.75}[fset] * {"cosine":0.9,"euclidean":0.85,"manhattan":0.8}[metric],
            })
            neighbors_cache[(fset, metric, k)] = (idxs, dists)

exp = pd.DataFrame(rows)
ranked = rank_models(exp, higher_is_better_cols=["avg_similarity_quality", "engagement_uplift", "interpretability"])
display(ranked.head(10))

  File "C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\univ\Data mining\Project\notebooks\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\LEGION\AppData\Local\Python\pythoncore-3.12-64\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executabl

,feature_set,metric,k,avg_similarity_quality,engagement_uplift,interpretability,composite_score
0,content-only,cosine,5,0.890780,-0.000424,0.855,2.614167
1,content-only,cosine,3,0.906867,-0.002155,0.855,2.529089
2,content-only,cosine,10,0.866160,-0.002263,0.855,2.473997
3,content-only,cosine,15,0.849307,-0.002264,0.855,2.453843
4,performance-only,cosine,3,0.978270,0.000831,0.765,2.441163
5,performance-only,cosine,5,0.972909,0.000780,0.765,2.431699
6,performance-only,cosine,15,0.953457,0.000540,0.765,2.394062
7,performance-only,cosine,10,0.962271,0.000046,0.765,2.374796
8,combined,cosine,3,0.860148,0.004161,0.675,2.147839
9,combined,cosine,5,0.845017,0.004101,0.675,2.126191


## Best Configuration and Business Recommendations

In [4]:
best = ranked.iloc[0]
best_key = (best["feature_set"], best["metric"], int(best["k"]))
idxs, dists = neighbors_cache[best_key]
k = int(best["k"])

recs = []
for i, row in df.reset_index(drop=True).iterrows():
    neigh = [j for j in idxs[i] if j != i][:k]
    top = df.iloc[neigh].sort_values("engagement_rate", ascending=False).head(max(1, len(neigh)//2))
    if top.empty:
        continue
    recs.append({
        "business_name": row["business_name"],
        "sector": row["sector"],
        "recommended_post_type": top["post_type"].mode().iat[0],
        "recommended_posting_time": top["posting_time_bin"].mode().iat[0],
        "recommended_caption_style": top["caption_length_bin"].mode().iat[0],
        "recommended_CTA_present": bool(top["CTA_present"].mean() >= 0.5),
        "recommended_mentions_location": bool(top["mentions_location"].mean() >= 0.5),
        "recommended_arabic_dialect_style": bool(top["arabic_dialect_style"].mean() >= 0.5),
        "avg_potential_uplift": float(top["engagement_rate"].mean() - row["engagement_rate"]),
    })

recommendations = pd.DataFrame(recs).groupby(["business_name","sector"], as_index=False).agg(
    recommended_post_type=("recommended_post_type", lambda x: x.mode().iat[0]),
    recommended_posting_time=("recommended_posting_time", lambda x: x.mode().iat[0]),
    recommended_caption_style=("recommended_caption_style", lambda x: x.mode().iat[0]),
    recommended_CTA_present=("recommended_CTA_present", "mean"),
    recommended_mentions_location=("recommended_mentions_location", "mean"),
    recommended_arabic_dialect_style=("recommended_arabic_dialect_style", "mean"),
    avg_potential_uplift=("avg_potential_uplift", "mean"),
)
for c in ["recommended_CTA_present","recommended_mentions_location","recommended_arabic_dialect_style"]:
    recommendations[c] = recommendations[c] >= 0.5

cluster_features = ["engagement_rate","view_rate","comment_rate","caption_length","hashtags_count","emoji_count"]
df["post_cluster"] = KMeans(n_clusters=5, random_state=42, n_init=20).fit_predict(df[cluster_features].fillna(0))
cluster_profile = df.groupby("post_cluster", as_index=False).agg(
    engagement_rate=("engagement_rate","mean"),
    top_post_type=("post_type", lambda x: x.mode().iat[0]),
    top_time=("posting_time_bin", lambda x: x.mode().iat[0]),
)

recommendations.to_csv(PROCESSED_DIR / "recommendations.csv", index=False)
ranked.to_csv(REPORTS_DIR / "recommendation_experiments.csv", index=False)
cluster_profile.to_csv(REPORTS_DIR / "recommendation_cluster_profiles.csv", index=False)
display(recommendations.head(15))
print("Best config:", best.to_dict())
print("Business insight: avg uplift =", round(recommendations["avg_potential_uplift"].mean(), 4))

,business_name,sector,recommended_post_type,recommended_posting_time,recommended_caption_style,recommended_CTA_present,recommended_mentions_location,recommended_arabic_dialect_style,avg_potential_uplift
0,Al Amal Dental,clinic,reel,afternoon,medium,True,True,True,0.175401
1,Al Balad Grill,restaurant,reel,afternoon,medium,True,True,True,0.060340
2,Al Hayat Pharmacy,pharmacy,reel,afternoon,medium,True,False,True,0.126375
3,Al Karmel Boutique,store,carousel,afternoon,medium,True,True,True,0.069250
4,Bethlehem Brew Bar,cafe,image,evening,medium,True,False,True,0.048135
5,Bethlehem Lash Bar,beauty_salon,image,afternoon,medium,True,True,True,0.099577
6,Canaan Threads,store,image,afternoon,medium,True,True,True,0.073631
7,Gaza Family Pharmacy,pharmacy,reel,afternoon,medium,True,True,True,0.144481
8,Gaza Gadget Shop,electronics_store,carousel,afternoon,medium,True,False,True,0.050228
9,Gaza Glow Salon,beauty_salon,carousel,afternoon,medium,True,True,True,0.085973


Best config: {'feature_set': 'content-only', 'metric': 'cosine', 'k': 5, 'avg_similarity_quality': 0.8907799999999999, 'engagement_uplift': -0.00042370369256980645, 'interpretability': 0.855, 'composite_score': 2.61416655280847}
Business insight: avg uplift = 0.0857
